# lab_08_jitter_integration

Goal
----
Turn a datasheet-style SSB phase-noise curve L(f) [dBc/Hz] into an rms timing
jitter, and verify the numerical integral against the 1/f^2 closed form.

Scenario (the canonical "手感" example):
    f0           = 5 GHz
    L(1 MHz)     = -100 dBc/Hz, assumed pure 1/f^2 skirt
    integrate from 1 MHz to 100 MHz

Closed form for a 1/f^2 skirt L(f) = L_ref*(f_ref/f)^2:
    sigma_phi^2 = integral 2*L(f) df = 2 L_ref_lin f_ref^2 (1/f1 - 1/f2)
    sigma_t     = sigma_phi / (2 pi f0)

Figure produced
---------------
  static/figures/phase_noise_to_jitter_integration.png

---

> 本 notebook 由 `scripts/make_notebooks.py` 從 `simulations/lab_08_jitter_integration.py` **自動產生**（generated snapshot，非手寫檔）。
> 權威版本是 repo 裡的 lab script；lab 更新後請重跑產生器同步。
> 執行需求：clone [isf-teaching-site](https://github.com/gmcycle7/isf-teaching-site)（要 import `simulations/common`）＋ `numpy` / `scipy` / `matplotlib`。

In [ ]:
# --- Setup：本 notebook 需要教學網站 repo 的 simulations/common 模組 ---
# 還沒有原始碼的話，先 clone repo，並把本 notebook 放在 repo 目錄樹內執行：
#     git clone https://github.com/gmcycle7/isf-teaching-site.git
# 相依套件只有三個：pip install numpy scipy matplotlib（外加 jupyter 本身）
import sys
from pathlib import Path

def _find_repo_root():
    """從目前工作目錄往上找，直到看到 simulations/common 為止。"""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "simulations" / "common").is_dir():
            return base
    raise FileNotFoundError(
        "找不到 simulations/common —— 請把本 notebook 放進 isf-teaching-site "
        "repo 目錄樹內執行（git clone https://github.com/gmcycle7/isf-teaching-site.git），"
        "或手動把 <repo>/simulations/common 加入 sys.path")

ROOT = _find_repo_root()
for _p in (str(ROOT), str(ROOT / "simulations" / "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root:", ROOT)

# CJK 字型：圖的標籤有繁體中文；找不到 CJK 字型只影響文字顯示、不影響任何數值
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm
_avail = {f.name for f in _fm.fontManager.ttflist}
_cjk = next((f for f in ["Heiti TC", "Arial Unicode MS", "STHeiti",
                         "Hiragino Sans GB", "Songti SC", "PingFang TC",
                         "Noto Sans CJK TC", "Microsoft JhengHei"]
             if f in _avail), None)
if _cjk:
    plt.rcParams["font.family"] = [_cjk, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False   # ASCII 減號，避免變方塊
print("CJK font:", _cjk or "(none found — 中文標籤可能顯示為方塊)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from noise_utils import leeson_one_over_f2, integrate_rms_jitter

# notebook 版 savefig：改成 inline 顯示。
# （原始 lab script 的 plot_utils.savefig 會把 PNG 寫進 static/figures/ 且從不
#   show()；在 notebook 裡我們直接把圖畫在 cell 輸出。）
def savefig(fig, name, verbose=True):
    plt.show()
    plt.close(fig)

In [ ]:
def main():
    print("[lab_08] phase noise -> rms jitter integration ...")
    f0 = 5e9
    f_ref = 1e6
    L_ref = -100.0  # dBc/Hz
    f1, f2 = 1e6, 100e6

    f = np.logspace(np.log10(f1), np.log10(f2), 4000)
    L = leeson_one_over_f2(f, L_ref, f_ref)

    # numerical integration
    sigma_t, sigma_phi = integrate_rms_jitter(f, L, f0, f1, f2)

    # analytic closed form (1/f^2)
    L_ref_lin = 10 ** (L_ref / 10)
    sigma_phi2_analytic = 2 * L_ref_lin * f_ref ** 2 * (1 / f1 - 1 / f2)
    sigma_phi_analytic = np.sqrt(sigma_phi2_analytic)
    sigma_t_analytic = sigma_phi_analytic / (2 * np.pi * f0)

    print(f"    sigma_phi (numeric)  = {sigma_phi*1e3:.4f} mrad")
    print(f"    sigma_phi (analytic) = {sigma_phi_analytic*1e3:.4f} mrad")
    print(f"    sigma_t   (numeric)  = {sigma_t*1e15:.2f} fs")
    print(f"    sigma_t   (analytic) = {sigma_t_analytic*1e15:.2f} fs")

    fig, ax = plt.subplots(figsize=(8.2, 5.2))
    ax.semilogx(f, L, color="tab:blue", lw=1.8, label=r"$L(f)$ (1/$f^2$ skirt)")
    ax.fill_between(f, L, L.min() - 5, alpha=0.12, color="tab:blue")
    ax.plot(f_ref, L_ref, "o", color="tab:red")
    ax.annotate(f"L(1 MHz) = {L_ref:.0f} dBc/Hz", xy=(f_ref, L_ref),
                xytext=(2e6, L_ref + 8),
                arrowprops=dict(arrowstyle="->", color="tab:red"))
    txt = (f"integrate {f1/1e6:.0f}–{f2/1e6:.0f} MHz\n"
           f"$\\sigma_\\phi$ = {sigma_phi*1e3:.2f} mrad\n"
           f"$\\sigma_t$ = {sigma_t*1e15:.1f} fs\n"
           f"(analytic {sigma_t_analytic*1e15:.1f} fs)")
    ax.text(0.62, 0.72, txt, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round", fc="white", ec="gray"))
    ax.set_xlabel("offset frequency $f$ [Hz]")
    ax.set_ylabel("$L(f)$ [dBc/Hz]")
    ax.set_title(r"Phase noise $\to$ rms jitter ($f_0$ = 5 GHz)")
    ax.legend(loc="lower left")
    savefig(fig, "phase_noise_to_jitter_integration.png")

In [ ]:
# 執行整個 lab（對應原 script 的 __main__）
main()